[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/08_knn/04_failure_cases_and_curse_of_dimensionality.ipynb)

# Bài 04 — Khi nào KNN thất bại? Curse of dimensionality và các failure case

**🎯 Mục tiêu**
- 5 failure case của KNN: nhiều chiều, feature vô ích, mất cân bằng lớp, dữ liệu lớn (chậm), ngoại suy.
- Cách khắc phục cho từng trường hợp.
- Hoàn thiện template model cho KNN.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/08_knn

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Failure case 1 — Nhiều chiều (curse of dimensionality)

Khi số chiều tăng, khoảng cách tới láng giềng gần nhất và xa nhất gần như bằng nhau (chương 05, bài 06) → "láng giềng gần nhất" mất ý nghĩa. Đo **tỉ lệ khoảng cách gần nhất / xa nhất** trung bình:

In [ ]:
ratios = []
dims = [1, 2, 5, 10, 20, 50, 100, 300]
for d in dims:
    X = rng.uniform(size=(1000, d))
    D = np.linalg.norm(X[:50, None, :] - X[None, :, :], axis=-1)
    D[np.arange(50), np.arange(50)] = np.nan
    ratios.append(np.nanmean(np.nanmin(D, 1) / np.nanmax(D, 1)))
plt.semilogx(dims, ratios, "o-")
plt.xlabel("số chiều d")
plt.ylabel("khoảng cách gần nhất / xa nhất")
plt.title("Trong nhiều chiều, 'gần nhất' và 'xa nhất' gần như bằng nhau")
plt.show()

## 2. Failure case 2 — Feature vô ích

Hai feature có ích + nhiều feature nhiễu: KNN suy giảm nhanh vì feature nhiễu "pha loãng" khoảng cách. **Khắc phục:** feature selection (chương 18), giảm chiều (PCA — chương 21), gán trọng số feature, hoặc học metric.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

n = 800
X_sig = rng.normal(size=(n, 2))
y = (X_sig[:, 0] ** 2 + X_sig[:, 1] ** 2 < 1.4).astype(int)     # ranh giới hình tròn — KNN rất hợp
rows = []
for n_noise in [0, 5, 20, 50]:
    X = np.hstack([X_sig, rng.normal(size=(n, n_noise))])
    plain = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)), X, y, cv=5).mean()
    selected = cross_val_score(make_pipeline(StandardScaler(), SelectKBest(mutual_info_classif, k=2),
                                             KNeighborsClassifier(15)), X, y, cv=5).mean()
    rows.append((n_noise, plain, selected))
    print(f"{n_noise:3d} feature nhiễu: KNN = {plain:.3f} | KNN + chọn 2 feature (trong CV) = {selected:.3f}")

## 3. Failure case 3 — Mất cân bằng lớp

Lớp đa số áp đảo trong mọi vùng lân cận → lớp thiểu số hiếm khi thắng phiếu. **Khắc phục:** `weights="distance"`, chọn ngưỡng trên `predict_proba` thay vì bỏ phiếu 50%, resampling (chương 19).

In [ ]:
from sklearn.metrics import recall_score

from ml_course.datasets import make_fraud_transactions

fraud = make_fraud_transactions(n=15000, fraud_rate=0.03, seed=3)
cols = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
Xf = fraud[cols].astype(float).to_numpy()
Xf[:, 0] = np.log(Xf[:, 0])
yf = fraud["is_fraud"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(25)).fit(Xtr, ytr)
proba = knn.predict_proba(Xte)[:, 1]
for t in [0.5, 0.2, 0.08]:
    print(f"ngưỡng {t:4}: recall gian lận = {recall_score(yte, proba >= t):.2f} | số cảnh báo = {(proba >= t).sum()}")

## 4. Failure case 4 — Dữ liệu lớn: dự đoán chậm, tốn bộ nhớ

Mỗi dự đoán phải so với **toàn bộ** dữ liệu train. Thời gian dự đoán tăng tuyến tính theo $n$ — với dịch vụ cần trả lời trong vài mili-giây trên hàng triệu bản ghi, KNN brute force không khả thi.

In [ ]:
d = 30
times = []
sizes = [1_000, 10_000, 50_000, 100_000]
for n_train in sizes:
    Xb = rng.normal(size=(n_train, d))
    model = KNeighborsClassifier(5, algorithm="brute").fit(Xb, rng.integers(0, 2, n_train))
    q = rng.normal(size=(200, d))
    start = time.perf_counter()
    model.predict(q)
    times.append((time.perf_counter() - start) / 200 * 1000)
plt.loglog(sizes, times, "o-")
plt.xlabel("số mẫu train")
plt.ylabel("ms / dự đoán")
plt.title("Thời gian dự đoán tăng tuyến tính theo kích thước dữ liệu")
plt.show()

**Khắc phục:** approximate nearest neighbor (FAISS, HNSW, Annoy), giảm số mẫu (prototype selection, condensed NN), giảm chiều, hoặc chuyển sang model "học" được (tree, linear, neural net) cho dự đoán nhanh.

## 5. Failure case 5 — Ngoại suy

KNN regressor dự đoán bằng trung bình các điểm đã thấy → **không bao giờ** dự đoán ngoài khoảng giá trị $y$ của tập train.

In [ ]:
x = np.sort(rng.uniform(0, 10, 100))
yy = 3 * x + rng.normal(0, 1, 100)
reg = KNeighborsRegressor(5).fit(x[:, None], yy)
for v in [5, 10, 15, 30]:
    print(f"x = {v:2d}: KNN dự đoán {reg.predict([[v]])[0]:6.2f} | quy luật thật {3 * v}")

## 6. Template model: KNN

| Phần | KNN |
|---|---|
| Problem | phân loại, hồi quy (cũng dùng cho tìm kiếm tương tự, recommender, impute) |
| Intuition | điểm gần nhau có nhãn giống nhau |
| Geometry | Voronoi ($k = 1$); ranh giới phi tuyến bất kỳ |
| Probability | ước lượng mật độ cục bộ: $P(y = c \mid \mathbf x) \approx$ tỉ lệ lớp $c$ trong $k$ láng giềng |
| Objective / Optimization | không có — lazy learning |
| Assumptions | độ đo khoảng cách có ý nghĩa; hàm mục tiêu "trơn" cục bộ; đủ dữ liệu phủ không gian |
| Complexity | train $O(1)$; dự đoán $O(nd)$ (brute) — chậm khi $n$ lớn; bộ nhớ $O(nd)$ |
| Hyperparameters | `n_neighbors`, `weights`, `metric`/`p`, `algorithm` |
| Preprocessing | **bắt buộc scaling**; xử lý missing; feature selection |
| Strengths | đơn giản, không giả định dạng hàm, ranh giới phức tạp, thêm dữ liệu không cần train lại, đa lớp tự nhiên |
| Weaknesses | chậm khi dự đoán, tốn bộ nhớ, curse of dimensionality, nhạy feature vô ích & thang đo, không ngoại suy |
| Interpretation | "dự đoán vì giống các ví dụ này" — giải thích bằng ví dụ |

## 🧠 Kiểm tra nhanh

1. Vì sao KNN suy giảm nhanh hơn Logistic Regression khi thêm feature nhiễu?
2. Có cách nào để KNN phát hiện gian lận tốt hơn mà không đổi model?
3. Một dịch vụ cần dự đoán < 5 ms với 50 triệu bản ghi lịch sử. KNN brute force có khả thi không? Đề xuất?

## 📌 Tóm tắt

KNN mạnh khi: ít chiều, đủ dữ liệu, feature đã scale và đều có ích, ranh giới phức tạp. Yếu khi: nhiều chiều, nhiều feature vô ích, mất cân bằng, dữ liệu rất lớn, cần ngoại suy.

## 📚 Tài liệu tham khảo

- Beyer et al. (1999) — *When Is "Nearest Neighbor" Meaningful?*
- Malkov & Yashunin (2018) — *HNSW: Efficient and robust approximate nearest neighbor search*
- scikit-learn User Guide — Nearest Neighbors